<a href="https://colab.research.google.com/github/WhySoSerious404/trust-safety-analytics/blob/main/02-abuse-detection-sql/abuse_detection_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Abuse detection in SQL

Generate synthetic platform activity, detect planted patterns
using SQL, and evaluate account-level flags.

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_DIR = Path(
    "/content/drive/MyDrive/Project: trust-safety/"
    "02-abuse-detection-sql/data"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Data folder:', DATA_DIR)

Mounted at /content/drive
Data folder: /content/drive/MyDrive/Project: trust-safety/02-abuse-detection-sql/data


## Synthetic data generator

In [3]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

N_ACCOUNTS = 20000
TARGET_EVENTS = 300000

START = pd.Timestamp("2026-09-01", tz="UTC")
END = START + pd.Timedelta(days=30)

PATTERN_COUNTS = {
    "legitimate": 19000,
    "spam": 400,
    "takeover": 300,
    "automated_clicking": 300
}

assert sum(PATTERN_COUNTS.values()) == N_ACCOUNTS

print("Accounts:", N_ACCOUNTS)
print("Target events:", TARGET_EVENTS)
print("Observation period:", START, "to", END, "(end excluded)")

pd.Series(PATTERN_COUNTS, name="account_count")

Accounts: 20000
Target events: 300000
Observation period: 2026-09-01 00:00:00+00:00 to 2026-10-01 00:00:00+00:00 (end excluded)


,account_count
legitimate,19000
spam,400
takeover,300
automated_clicking,300


In [4]:
account_ids = np.arange(1, N_ACCOUNTS + 1)

signup_dates = START - pd.to_timedelta(
    rng.integers(1, 61, size=N_ACCOUNTS),
    unit="D"
)

accounts = pd.DataFrame({
    "account_id": account_ids,
    "signup_date": signup_dates.strftime("%Y-%m-%d"),
    "country": rng.choice(
        ["IN", "US", "GB", "DE", "CA"],
        size=N_ACCOUNTS
    )
})

labels = np.repeat(
    list(PATTERN_COUNTS.keys()),
    list(PATTERN_COUNTS.values())
)

rng.shuffle(labels)

ground_truth = pd.DataFrame({
    "account_id": account_ids,
    "pattern": labels
})

assert accounts["account_id"].is_unique
assert len(accounts) == N_ACCOUNTS

display(accounts.head())
ground_truth["pattern"].value_counts()

,account_id,signup_date,country
0,1,2026-08-26,IN
1,2,2026-07-16,DE
2,3,2026-07-23,US
3,4,2026-08-05,DE
4,5,2026-08-06,GB


,count
pattern,
legitimate,19000
spam,400
takeover,300
automated_clicking,300


In [5]:
base_account_ids = np.repeat(
    accounts["account_id"].to_numpy(),
    10
)

extra_count = TARGET_EVENTS - len(base_account_ids)

extra_account_ids = rng.choice(
    accounts["account_id"].to_numpy(),
    size=extra_count
)

event_account_ids = np.concatenate([
    base_account_ids,
    extra_account_ids
])

seconds_in_period = int((END - START).total_seconds())

event_offsets = rng.integers(
    0,
    seconds_in_period,
    size=TARGET_EVENTS
)

events = pd.DataFrame({
    "account_id": event_account_ids,
    "event_time": START + pd.to_timedelta(event_offsets, unit="s"),
    "event_type": rng.choice(
        ["login", "post", "click"],
        size=TARGET_EVENTS,
        p=[0.3, 0.2, 0.5]
    )
})

country_lookup = accounts.set_index("account_id")["country"]
events["country"] = events["account_id"].map(country_lookup)

print("Baseline events:", len(events))
print("Minimum events per account:",
      events.groupby("account_id").size().min())

events.head()

Baseline events: 300000
Minimum events per account: 10


,account_id,event_time,event_type,country
0,1,2026-09-23 17:53:31+00:00,click,IN
1,1,2026-09-28 12:05:56+00:00,click,IN
2,1,2026-09-02 17:53:48+00:00,login,IN
3,1,2026-09-09 14:44:31+00:00,post,IN
4,1,2026-09-30 07:48:46+00:00,login,IN


In [6]:
ip_lookup = {}

for account_id in accounts["account_id"]:
    a, b, c = rng.integers(1, 255, size=3)
    ip_lookup[account_id] = f"10.{a}.{b}.{c}"

events["ip_address"] = events["account_id"].map(ip_lookup)

assert events["ip_address"].notna().all()

events.head()

,account_id,event_time,event_type,country,ip_address
0,1,2026-09-23 17:53:31+00:00,click,IN,10.145.131.65
1,1,2026-09-28 12:05:56+00:00,click,IN,10.145.131.65
2,1,2026-09-02 17:53:48+00:00,login,IN,10.145.131.65
3,1,2026-09-09 14:44:31+00:00,post,IN,10.145.131.65
4,1,2026-09-30 07:48:46+00:00,login,IN,10.145.131.65


## Constructing the simulation

### Spam IDs

In [7]:
spam_ids = ground_truth.loc[
    ground_truth["pattern"] == "spam",
    "account_id"
].to_numpy()

rng.shuffle(spam_ids)

spam_gangs = np.array_split(spam_ids, 20)

print("Planted spam accounts:", len(spam_ids))
print("Number of gangs:", len(spam_gangs))
print("Accounts per gang:", len(spam_gangs[0]))
print("First gang:", spam_gangs[0])

Planted spam accounts: 400
Number of gangs: 20
Accounts per gang: 20
First gang: [  339   202  9086  1596 10360 16164  5010 18614  5333  7844  8127 12664
  1291 12808  3930 10754  4544 15316  2621  9276]


In [8]:
account_event_rows = events.groupby("account_id").groups

changed_spam_rows = []

for gang_number, gang in enumerate(spam_gangs, start=1):
    start_offset = int(
        rng.integers(0, seconds_in_period - 600)
    )

    gang_start = START + pd.Timedelta(seconds=start_offset)

    for account_id in gang:
        account_rows = account_event_rows[account_id]

        max_posts = min(12, len(account_rows))
        n_posts = int(rng.integers(2, max_posts + 1))

        selected_rows = rng.choice(
            account_rows,
            size=n_posts,
            replace=False
        )

        events.loc[selected_rows, "event_type"] = "post"

        events.loc[selected_rows, "event_time"] = (
            gang_start
            + pd.to_timedelta(
                rng.integers(0, 600, size=n_posts),
                unit="s"
            )
        )

        last_octet = int(rng.integers(1, 255))

        events.loc[selected_rows, "ip_address"] = (
            f"10.250.{gang_number}.{last_octet}"
        )

        changed_spam_rows.extend(selected_rows.tolist())

assert len(events) == TARGET_EVENTS

print("Events modified:", len(changed_spam_rows))
print("Total events:", len(events))

display(events.loc[changed_spam_rows[:5]])

Events modified: 2885
Total events: 300000


,account_id,event_time,event_type,country,ip_address
3380,339,2026-09-27 15:33:16+00:00,post,IN,10.250.1.34
239094,339,2026-09-27 15:41:52+00:00,post,IN,10.250.1.34
3385,339,2026-09-27 15:34:12+00:00,post,IN,10.250.1.34
3386,339,2026-09-27 15:41:45+00:00,post,IN,10.250.1.34
3382,339,2026-09-27 15:33:44+00:00,post,IN,10.250.1.34


In [9]:
spam_check = events.loc[changed_spam_rows]

assert spam_check.index.is_unique
assert set(spam_check["account_id"]) == set(spam_ids)
assert spam_check["event_type"].eq("post").all()

gang_checks = []

for gang_number, gang in enumerate(spam_gangs, start=1):
    gang_posts = spam_check[
        spam_check["account_id"].isin(gang)
    ]

    span_seconds = (
        gang_posts["event_time"].max()
        - gang_posts["event_time"].min()
    ).total_seconds()

    ip_blocks = (
        gang_posts["ip_address"]
        .str.rsplit(".", n=1)
        .str[0]
    )

    assert span_seconds < 600
    assert ip_blocks.nunique() == 1

    gang_checks.append({
        "gang": gang_number,
        "accounts": gang_posts["account_id"].nunique(),
        "posts": len(gang_posts),
        "span_seconds": span_seconds,
        "ip_block": ip_blocks.iloc[0]
    })

print("Spam accounts represented:", spam_check["account_id"].nunique())
print("Gangs checked:", len(gang_checks))

pd.DataFrame(gang_checks).head()

Spam accounts represented: 400
Gangs checked: 20


,gang,accounts,posts,span_seconds,ip_block
0,1,20,155,594.0,10.250.1
1,2,20,133,597.0,10.250.2
2,3,20,138,593.0,10.250.3
3,4,20,150,599.0,10.250.4
4,5,20,148,592.0,10.250.5


In [10]:
spam_check = events.loc[changed_spam_rows]

assert spam_check.index.is_unique
assert set(spam_check["account_id"]) == set(spam_ids)
assert spam_check["event_type"].eq("post").all()

gang_checks = []

for gang_number, gang in enumerate(spam_gangs, start=1):
    gang_posts = spam_check[
        spam_check["account_id"].isin(gang)
    ]

    span_seconds = (
        gang_posts["event_time"].max()
        - gang_posts["event_time"].min()
    ).total_seconds()

    ip_blocks = (
        gang_posts["ip_address"]
        .str.rsplit(".", n=1)
        .str[0]
    )

    assert span_seconds < 600
    assert ip_blocks.nunique() == 1

    gang_checks.append({
        "gang": gang_number,
        "accounts": gang_posts["account_id"].nunique(),
        "posts": len(gang_posts),
        "span_seconds": span_seconds,
        "ip_block": ip_blocks.iloc[0]
    })

print("Spam accounts represented:", spam_check["account_id"].nunique())
print("Gangs checked:", len(gang_checks))

pd.DataFrame(gang_checks).head()

Spam accounts represented: 400
Gangs checked: 20


,gang,accounts,posts,span_seconds,ip_block
0,1,20,155,594.0,10.250.1
1,2,20,133,597.0,10.250.2
2,3,20,138,593.0,10.250.3
3,4,20,150,599.0,10.250.4
4,5,20,148,592.0,10.250.5


### Takeover IDs

In [11]:
takeover_ids = ground_truth.loc[
    ground_truth["pattern"] == "takeover",
    "account_id"
].to_numpy()

event_counts = events.groupby("account_id").size()

print("Planted takeover accounts:", len(takeover_ids))
print(
    "Minimum available events:",
    event_counts.loc[takeover_ids].min()
)

Planted takeover accounts: 300
Minimum available events: 10


In [12]:
countries = accounts["country"].unique()

takeover_plan_rows = []

for account_id in takeover_ids:
    home_country = country_lookup.loc[account_id]

    other_countries = [
        country
        for country in countries
        if country != home_country
    ]

    start_offset = int(
        rng.integers(
            7 * 24 * 60 * 60,
            seconds_in_period - 3600
        )
    )

    takeover_plan_rows.append({
        "account_id": account_id,
        "home_country": home_country,
        "new_country": rng.choice(other_countries),
        "takeover_time": START + pd.Timedelta(
            seconds=start_offset
        ),
        "burst_seconds": int(rng.choice([600, 1800, 3600]))
    })

takeover_plan = pd.DataFrame(takeover_plan_rows)

assert takeover_plan["home_country"].ne(
    takeover_plan["new_country"]
).all()

print("Takeover plans:", len(takeover_plan))
takeover_plan.head()

Takeover plans: 300


,account_id,home_country,new_country,takeover_time,burst_seconds
0,31,CA,GB,2026-09-19 04:33:52+00:00,1800
1,34,DE,US,2026-09-22 15:38:57+00:00,3600
2,41,DE,CA,2026-09-25 03:31:22+00:00,1800
3,103,DE,GB,2026-09-09 21:54:01+00:00,1800
4,132,CA,DE,2026-09-25 15:48:38+00:00,1800


In [13]:
changed_takeover_rows = []
takeover_history_rows = []

for _, plan in takeover_plan.iterrows():
    account_id = plan["account_id"]

    account_rows = rng.permutation(
        account_event_rows[account_id]
    )

    history_rows = account_rows[:6]
    attack_rows = account_rows[6:]

    events.loc[history_rows, "event_time"] = (
        START
        + pd.to_timedelta(
            rng.integers(0, 7 * 24 * 60 * 60, size=6),
            unit="s"
        )
    )

    events.loc[history_rows, "country"] = plan["home_country"]
    events.loc[history_rows, "ip_address"] = ip_lookup[account_id]
    events.loc[history_rows[0], "event_type"] = "login"

    attack_offsets = rng.integers(
        1,
        int(plan["burst_seconds"]),
        size=len(attack_rows)
    )
    attack_offsets[0] = 0

    events.loc[attack_rows, "event_time"] = (
        plan["takeover_time"]
        + pd.to_timedelta(attack_offsets, unit="s")
    )

    events.loc[attack_rows, "country"] = plan["new_country"]

    events.loc[attack_rows, "event_type"] = rng.choice(
        ["post", "click"],
        size=len(attack_rows)
    )
    events.loc[attack_rows[0], "event_type"] = "login"

    a, b, c = rng.integers(1, 255, size=3)
    events.loc[attack_rows, "ip_address"] = f"10.{a}.{b}.{c}"

    takeover_history_rows.extend(history_rows.tolist())
    changed_takeover_rows.extend(attack_rows.tolist())

assert len(events) == TARGET_EVENTS

print("History events:", len(takeover_history_rows))
print("Takeover accounts:",
      events.loc[changed_takeover_rows, "account_id"].nunique())
print("Total events:", len(events))

example_id = takeover_ids[0]

display(
    events.loc[events["account_id"] == example_id]
    .sort_values("event_time")
)

History events: 1800
Takeover accounts: 300
Total events: 300000


,account_id,event_time,event_type,country,ip_address
277911,31,2026-09-02 01:25:00+00:00,post,CA,10.208.75.175
309,31,2026-09-02 05:57:11+00:00,click,CA,10.208.75.175
304,31,2026-09-02 16:52:09+00:00,post,CA,10.208.75.175
301,31,2026-09-06 00:25:14+00:00,login,CA,10.208.75.175
303,31,2026-09-07 14:36:59+00:00,click,CA,10.208.75.175
295427,31,2026-09-07 19:35:06+00:00,click,CA,10.208.75.175
300,31,2026-09-19 04:33:52+00:00,login,GB,10.164.109.104
307,31,2026-09-19 04:39:14+00:00,post,GB,10.164.109.104
259841,31,2026-09-19 04:39:54+00:00,post,GB,10.164.109.104
308,31,2026-09-19 04:39:56+00:00,click,GB,10.164.109.104


In [14]:
for ri, plan in takeover_plan.iterrows():
    activity = events.loc[
        account_event_rows[plan["account_id"]]
    ]

    history = activity[
        activity["event_time"] < plan["takeover_time"]
    ]

    attack = activity[
        activity["event_time"] >= plan["takeover_time"]
    ]

    assert len(history) == 6
    assert history["country"].eq(plan["home_country"]).all()
    assert history["event_type"].eq("login").any()

    assert attack["country"].eq(plan["new_country"]).all()

    burst_end = plan["takeover_time"] + pd.Timedelta(
        seconds=int(plan["burst_seconds"])
    )

    assert attack["event_time"].lt(burst_end).all()

    first_attack = attack.sort_values("event_time").iloc[0]

    assert first_attack["event_time"] == plan["takeover_time"]
    assert first_attack["event_type"] == "login"

print("Verified takeover accounts:", len(takeover_plan))

Verified takeover accounts: 300


### Click IDs

In [15]:
click_ids = ground_truth.loc[
    ground_truth["pattern"] == "automated_clicking",
    "account_id"
].to_numpy()

changed_click_rows = []
click_plan_rows = []

for account_id in click_ids:
    account_rows = account_event_rows[account_id]

    max_clicks = min(14, len(account_rows))
    n_clicks = int(rng.integers(4, max_clicks + 1))

    burst_seconds = int(rng.choice([10, 30, 120]))

    start_offset = int(
        rng.integers(0, seconds_in_period - burst_seconds)
    )
    burst_start = START + pd.Timedelta(seconds=start_offset)

    selected_rows = rng.choice(
        account_rows,
        size=n_clicks,
        replace=False
    )

    events.loc[selected_rows, "event_type"] = "click"

    events.loc[selected_rows, "event_time"] = (
        burst_start
        + pd.to_timedelta(
            rng.integers(0, burst_seconds, size=n_clicks),
            unit="s"
        )
    )

    changed_click_rows.extend(selected_rows.tolist())

    click_plan_rows.append({
        "account_id": account_id,
        "burst_start": burst_start,
        "burst_seconds": burst_seconds,
        "modified_clicks": n_clicks
    })

click_plan = pd.DataFrame(click_plan_rows)

assert len(events) == TARGET_EVENTS

print("Planted clicking accounts:", len(click_ids))
print("Events modified:", len(changed_click_rows))
print("Total events:", len(events))

display(click_plan.head())

Planted clicking accounts: 300
Events modified: 2609
Total events: 300000


,account_id,burst_start,burst_seconds,modified_clicks
0,66,2026-09-08 03:21:59+00:00,10,12
1,100,2026-09-25 04:18:43+00:00,10,5
2,204,2026-09-27 22:57:57+00:00,10,4
3,290,2026-09-09 00:10:14+00:00,10,5
4,313,2026-09-11 19:39:33+00:00,120,11


In [16]:
click_check = events.loc[changed_click_rows]

assert click_check.index.is_unique
assert set(click_check["account_id"]) == set(click_ids)
assert click_check["event_type"].eq("click").all()
assert len(click_check) == click_plan["modified_clicks"].sum()

for plan in click_plan.itertuples(index=False):
    account_clicks = click_check[
        click_check["account_id"] == plan.account_id
    ]

    burst_end = plan.burst_start + pd.Timedelta(
        seconds=plan.burst_seconds
    )

    assert len(account_clicks) == plan.modified_clicks
    assert account_clicks["event_time"].ge(plan.burst_start).all()
    assert account_clicks["event_time"].lt(burst_end).all()

print("Verified clicking accounts:", len(click_plan))

Verified clicking accounts: 300


### Legitimate lookalike IDs

In [17]:
legitimate_ids = ground_truth.loc[
    ground_truth["pattern"] == "legitimate",
    "account_id"
].to_numpy()

lookalike_ids = rng.choice(
    legitimate_ids,
    size=800,
    replace=False
)

legit_shared_ids = lookalike_ids[:400]
legit_travel_ids = lookalike_ids[400:600]
legit_click_ids = lookalike_ids[600:]

assert len(np.unique(lookalike_ids)) == 800
assert set(lookalike_ids).issubset(set(legitimate_ids))

print("Shared-network accounts:", len(legit_shared_ids))
print("Travel accounts:", len(legit_travel_ids))
print("Genuine-click accounts:", len(legit_click_ids))

Shared-network accounts: 400
Travel accounts: 200
Genuine-click accounts: 200


In [18]:
legit_shared_groups = np.array_split(legit_shared_ids, 20)

block_numbers = rng.permutation(np.arange(1, 21))

changed_legit_shared_rows = []

for group, block_number in zip(
    legit_shared_groups,
    block_numbers
):
    start_offset = int(
        rng.integers(0, seconds_in_period - 600)
    )
    group_start = START + pd.Timedelta(seconds=start_offset)

    for account_id in group:
        account_rows = account_event_rows[account_id]
        n_posts = int(rng.integers(1, 6))

        selected_rows = rng.choice(
            account_rows,
            size=n_posts,
            replace=False
        )

        events.loc[selected_rows, "event_type"] = "post"

        events.loc[selected_rows, "event_time"] = (
            group_start
            + pd.to_timedelta(
                rng.integers(0, 600, size=n_posts),
                unit="s"
            )
        )

        last_octet = int(rng.integers(1, 255))

        events.loc[selected_rows, "ip_address"] = (
            f"10.250.{block_number}.{last_octet}"
        )

        changed_legit_shared_rows.extend(selected_rows.tolist())

assert len(events) == TARGET_EVENTS

print(
    "Legitimate shared-network accounts:",
    events.loc[changed_legit_shared_rows, "account_id"].nunique()
)
print("Events modified:", len(changed_legit_shared_rows))
print("Total events:", len(events))

display(events.loc[changed_legit_shared_rows[:5]])

Legitimate shared-network accounts: 400
Events modified: 1184
Total events: 300000


,account_id,event_time,event_type,country,ip_address
295061,11048,2026-09-21 02:31:22+00:00,post,IN,10.250.7.35
111187,11119,2026-09-21 02:35:47+00:00,post,DE,10.250.7.13
111181,11119,2026-09-21 02:29:01+00:00,post,DE,10.250.7.13
111180,11119,2026-09-21 02:31:53+00:00,post,DE,10.250.7.13
5085,509,2026-09-21 02:27:36+00:00,post,GB,10.250.7.51


In [19]:
busy_travel_ids = set(
    rng.choice(legit_travel_ids, size=50, replace=False)
)

legit_travel_plan_rows = []

for account_id in legit_travel_ids:
    home_country = country_lookup.loc[account_id]

    other_countries = [
        country for country in countries
        if country != home_country
    ]

    travel_offset = int(
        rng.integers(7 * 24 * 60 * 60, 20 * 24 * 60 * 60)
    )
    travel_time = START + pd.Timedelta(seconds=travel_offset)

    busy_visit = account_id in busy_travel_ids

    activity_window = (
        1800 if busy_visit
        else int((END - travel_time).total_seconds())
    )

    legit_travel_plan_rows.append({
        "account_id": account_id,
        "home_country": home_country,
        "new_country": rng.choice(other_countries),
        "travel_time": travel_time,
        "activity_window_seconds": activity_window,
        "busy_visit": busy_visit
    })

legit_travel_plan = pd.DataFrame(legit_travel_plan_rows)

print("Legitimate travelers:", len(legit_travel_plan))
print("Busy travelers:", legit_travel_plan["busy_visit"].sum())

display(legit_travel_plan.head())

Legitimate travelers: 200
Busy travelers: 50


,account_id,home_country,new_country,travel_time,activity_window_seconds,busy_visit
0,1900,US,DE,2026-09-10 22:04:07+00:00,1800,True
1,16643,DE,GB,2026-09-13 05:22:27+00:00,1800,True
2,18800,GB,CA,2026-09-18 04:02:58+00:00,1108622,False
3,1975,GB,IN,2026-09-19 11:48:32+00:00,1800,True
4,8295,US,GB,2026-09-20 23:15:45+00:00,866655,False


In [20]:
changed_legit_travel_rows = []
legit_travel_history_rows = []

for plan in legit_travel_plan.itertuples(index=False):
    account_id = plan.account_id

    account_rows = rng.permutation(
        account_event_rows[account_id]
    )

    history_rows = account_rows[:6]
    travel_rows = account_rows[6:]

    events.loc[history_rows, "event_time"] = (
        START
        + pd.to_timedelta(
            rng.integers(0, 7 * 24 * 60 * 60, size=6),
            unit="s"
        )
    )

    events.loc[history_rows, "country"] = plan.home_country
    events.loc[history_rows, "ip_address"] = ip_lookup[account_id]
    events.loc[history_rows[0], "event_type"] = "login"

    travel_offsets = rng.integers(
        1,
        plan.activity_window_seconds,
        size=len(travel_rows)
    )
    travel_offsets[0] = 0

    events.loc[travel_rows, "event_time"] = (
        plan.travel_time
        + pd.to_timedelta(travel_offsets, unit="s")
    )

    events.loc[travel_rows, "country"] = plan.new_country

    events.loc[travel_rows, "event_type"] = rng.choice(
        ["post", "click"],
        size=len(travel_rows)
    )
    events.loc[travel_rows[0], "event_type"] = "login"

    a, b, c = rng.integers(1, 255, size=3)
    events.loc[travel_rows, "ip_address"] = f"10.{a}.{b}.{c}"

    legit_travel_history_rows.extend(history_rows.tolist())
    changed_legit_travel_rows.extend(travel_rows.tolist())

assert len(events) == TARGET_EVENTS

print(
    "Legitimate travelers modified:",
    events.loc[changed_legit_travel_rows, "account_id"].nunique()
)
print("Total events:", len(events))

example_id = legit_travel_ids[0]

display(
    events.loc[events["account_id"] == example_id]
    .sort_values("event_time")
)

Legitimate travelers modified: 200
Total events: 300000


,account_id,event_time,event_type,country,ip_address
207655,1900,2026-09-01 21:58:09+00:00,login,US,10.135.44.183
18999,1900,2026-09-02 00:17:03+00:00,click,US,10.135.44.183
18993,1900,2026-09-02 15:10:14+00:00,click,US,10.135.44.183
18992,1900,2026-09-04 12:10:55+00:00,login,US,10.135.44.183
18997,1900,2026-09-04 12:50:48+00:00,login,US,10.135.44.183
18990,1900,2026-09-05 01:38:13+00:00,login,US,10.135.44.183
18996,1900,2026-09-10 22:04:07+00:00,login,DE,10.170.236.107
18995,1900,2026-09-10 22:10:12+00:00,post,DE,10.170.236.107
18994,1900,2026-09-10 22:22:44+00:00,click,DE,10.170.236.107
237100,1900,2026-09-10 22:22:57+00:00,post,DE,10.170.236.107


In [21]:
changed_legit_click_rows = []
legit_click_plan_rows = []

for account_id in legit_click_ids:
    account_rows = account_event_rows[account_id]

    max_clicks = min(14, len(account_rows))
    n_clicks = int(rng.integers(4, max_clicks + 1))

    burst_seconds = int(rng.choice([10, 30, 120]))

    start_offset = int(
        rng.integers(0, seconds_in_period - burst_seconds)
    )
    burst_start = START + pd.Timedelta(seconds=start_offset)

    selected_rows = rng.choice(
        account_rows,
        size=n_clicks,
        replace=False
    )

    events.loc[selected_rows, "event_type"] = "click"

    events.loc[selected_rows, "event_time"] = (
        burst_start
        + pd.to_timedelta(
            rng.integers(0, burst_seconds, size=n_clicks),
            unit="s"
        )
    )

    changed_legit_click_rows.extend(selected_rows.tolist())

    legit_click_plan_rows.append({
        "account_id": account_id,
        "burst_start": burst_start,
        "burst_seconds": burst_seconds,
        "modified_clicks": n_clicks
    })

legit_click_plan = pd.DataFrame(legit_click_plan_rows)

assert len(events) == TARGET_EVENTS

print(
    "Legitimate clicking accounts:",
    events.loc[changed_legit_click_rows, "account_id"].nunique()
)
print("Events modified:", len(changed_legit_click_rows))
print("Total events:", len(events))

display(legit_click_plan.head())

Legitimate clicking accounts: 200
Events modified: 1756
Total events: 300000


,account_id,burst_start,burst_seconds,modified_clicks
0,17472,2026-09-07 17:50:37+00:00,120,8
1,1263,2026-09-09 23:48:54+00:00,120,4
2,7660,2026-09-24 10:49:01+00:00,30,10
3,8910,2026-09-08 05:34:28+00:00,10,11
4,12474,2026-09-02 04:55:32+00:00,10,13


In [22]:
assert len(accounts) == N_ACCOUNTS
assert len(events) == TARGET_EVENTS

assert accounts["account_id"].is_unique
assert ground_truth["account_id"].is_unique

assert set(events["account_id"]) == set(accounts["account_id"])
assert set(ground_truth["account_id"]) == set(accounts["account_id"])

assert accounts.notna().all().all()
assert events.notna().all().all()
assert ground_truth.notna().all().all()

assert events["event_type"].isin(["login", "post", "click"]).all()
assert events["country"].isin(countries).all()

assert events["event_time"].ge(START).all()
assert events["event_time"].lt(END).all()

signup_lookup = pd.to_datetime(
    accounts.set_index("account_id")["signup_date"],
    utc=True
)

event_signup_dates = events["account_id"].map(signup_lookup)

assert events["event_time"].ge(event_signup_dates).all()

assert events.groupby("account_id").size().ge(10).all()

assert ground_truth["pattern"].value_counts().to_dict() == PATTERN_COUNTS
assert "pattern" not in events.columns

print("Structural checks passed.")
print("Accounts:", len(accounts))
print("Events:", len(events))
print("Ground-truth rows:", len(ground_truth))

Structural checks passed.
Accounts: 20000
Events: 300000
Ground-truth rows: 20000


In [23]:
# Shared-network lookalikes
shared_check = events.loc[changed_legit_shared_rows]

assert shared_check.index.is_unique
assert set(shared_check["account_id"]) == set(legit_shared_ids)
assert shared_check["event_type"].eq("post").all()

for group in legit_shared_groups:
    posts = shared_check[
        shared_check["account_id"].isin(group)
    ]

    span = (
        posts["event_time"].max()
        - posts["event_time"].min()
    ).total_seconds()

    blocks = posts["ip_address"].str.rsplit(".", n=1).str[0]

    assert span < 600
    assert blocks.nunique() == 1

print("Shared-network accounts verified:", len(legit_shared_ids))


# Travel lookalikes
for plan in legit_travel_plan.itertuples(index=False):
    activity = events.loc[account_event_rows[plan.account_id]]

    history = activity[activity["event_time"] < plan.travel_time]
    travel = activity[activity["event_time"] >= plan.travel_time]

    assert len(history) == 6
    assert history["country"].eq(plan.home_country).all()
    assert history["event_type"].eq("login").any()
    assert travel["country"].eq(plan.new_country).all()

    window_end = plan.travel_time + pd.Timedelta(
        seconds=plan.activity_window_seconds
    )
    assert travel["event_time"].lt(window_end).all()

    first = travel.sort_values("event_time").iloc[0]
    assert first["event_time"] == plan.travel_time
    assert first["event_type"] == "login"

print("Travel accounts verified:", len(legit_travel_plan))


# Genuine click spikes
click_check = events.loc[changed_legit_click_rows]

assert click_check.index.is_unique
assert set(click_check["account_id"]) == set(legit_click_ids)
assert click_check["event_type"].eq("click").all()

for plan in legit_click_plan.itertuples(index=False):
    clicks = click_check[
        click_check["account_id"] == plan.account_id
    ]

    window_end = plan.burst_start + pd.Timedelta(
        seconds=plan.burst_seconds
    )

    assert len(clicks) == plan.modified_clicks
    assert clicks["event_time"].ge(plan.burst_start).all()
    assert clicks["event_time"].lt(window_end).all()

print("Genuine-click accounts verified:", len(legit_click_plan))


# Labels remain legitimate
lookalike_labels = ground_truth.set_index("account_id").loc[
    lookalike_ids, "pattern"
]

assert lookalike_labels.eq("legitimate").all()

print("All lookalike checks passed.")

Shared-network accounts verified: 400
Travel accounts verified: 200
Genuine-click accounts verified: 200
All lookalike checks passed.


### Assigning Event IDs

In [24]:
events = events.sort_values(
    "event_time",
    kind="stable"
).reset_index(drop=True)

events["event_id"] = np.arange(1, len(events) + 1)

events = events[
    [
        "event_id",
        "account_id",
        "event_time",
        "event_type",
        "ip_address",
        "country"
    ]
]

assert events["event_id"].is_unique
assert events["event_time"].is_monotonic_increasing

accounts.to_csv(DATA_DIR / "accounts.csv", index=False)

events.to_csv(
    DATA_DIR / "events.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

ground_truth.to_csv(
    DATA_DIR / "ground_truth.csv",
    index=False
)

print("Saved to:", DATA_DIR)
print("Accounts:", len(accounts))
print("Events:", len(events))
print("Ground-truth rows:", len(ground_truth))

events.head()

Saved to: /content/drive/MyDrive/Project: trust-safety/02-abuse-detection-sql/data
Accounts: 20000
Events: 300000
Ground-truth rows: 20000


,event_id,account_id,event_time,event_type,ip_address,country
0,1,7651,2026-09-01 00:00:03+00:00,click,10.53.89.192,DE
1,2,5805,2026-09-01 00:00:05+00:00,click,10.85.62.234,GB
2,3,11491,2026-09-01 00:00:20+00:00,click,10.169.161.2,DE
3,4,18929,2026-09-01 00:00:27+00:00,click,10.240.241.64,US
4,5,12181,2026-09-01 00:00:38+00:00,click,10.37.101.144,DE


## Synthetic dataset summary

### Summary

Generated using seed 42 for September 1–30, 2026.
All event timestamps are UTC; October 1 is excluded.

### Tables
- accounts.csv: 20000 accounts.
- events.csv: 300000 events with unique, chronological event IDs.
- ground_truth.csv: one pattern label per account.

Ground-truth labels and generator plans are excluded from
SQL detection inputs.

### Planted accounts
- 19000 legitimate accounts.
- 400 spam accounts in 20 coordinated groups.
- 300 takeover accounts.
- 300 automated-clicking accounts.

Each planted abusive account belongs to one pattern.

### Legitimate lookalikes
Within the legitimate population, disjoint groups include:
- 400 accounts with coordinated shared-network posting.
- 200 travelers, including 50 with concentrated activity.
- 200 accounts with genuine click spikes.

Their labels remain legitimate. Some observable behaviour deliberately overlaps with planted abuse.

### Generator assumptions
Every account initially received ten events, with remaining
events randomly assigned. Baseline event-type probabilities
were 30% login, 20% post, and 50% click.

These probabilities, burst windows, and pattern counts are
design choices, not estimates of real platform behaviour.
Final event-type proportions differ after pattern injection.

### Validation
Checks passed for table sizes, account references, required
values, allowed event types, timestamps, signup ordering,
label counts, and generated pattern timelines.

These checks establish construction consistency, not realism
or detection effectiveness. Rule evaluation remains in-sample,
with no held-out dataset.

### Saved data
The three CSV files are stored in Google Drive under:
Project: trust-safety/02-abuse-detection-sql/data

SQL detection and evaluation have not yet been performed.

## SQL setup

### Initial Checks

In [25]:
import sqlite3

accounts_sql = pd.read_csv(DATA_DIR / "accounts.csv")
events_sql = pd.read_csv(DATA_DIR / "events.csv")

conn = sqlite3.connect(":memory:")

accounts_sql.to_sql(
    "accounts",
    conn,
    index=False,
    if_exists="replace"
)

events_sql.to_sql(
    "events",
    conn,
    index=False,
    if_exists="replace"
)

print("Accounts:", len(accounts_sql))
print("Events:", len(events_sql))

Accounts: 20000
Events: 300000


In [26]:
accounts_sql

,account_id,signup_date,country
0,1,2026-08-26,IN
1,2,2026-07-16,DE
2,3,2026-07-23,US
3,4,2026-08-05,DE
4,5,2026-08-06,GB
...,...,...,...
19995,19996,2026-08-22,CA
19996,19997,2026-08-22,US
19997,19998,2026-08-09,DE
19998,19999,2026-08-10,IN


In [27]:
events_sql

,event_id,account_id,event_time,event_type,ip_address,country
0,1,7651,2026-09-01 00:00:03,click,10.53.89.192,DE
1,2,5805,2026-09-01 00:00:05,click,10.85.62.234,GB
2,3,11491,2026-09-01 00:00:20,click,10.169.161.2,DE
3,4,18929,2026-09-01 00:00:27,click,10.240.241.64,US
4,5,12181,2026-09-01 00:00:38,click,10.37.101.144,DE
...,...,...,...,...,...,...
299995,299996,6746,2026-09-30 23:58:51,login,10.18.179.11,IN
299996,299997,18412,2026-09-30 23:59:07,click,10.117.168.233,DE
299997,299998,18899,2026-09-30 23:59:13,login,10.142.181.181,IN
299998,299999,5892,2026-09-30 23:59:24,login,10.51.44.104,GB


In [28]:
pd.read_sql_query("""
    SELECT country, COUNT(*) AS account_count FROM accounts
    GROUP BY country ORDER BY account_count DESC;
    """, conn
)

,country,account_count
0,DE,4090
1,CA,4033
2,IN,3979
3,US,3961
4,GB,3937


In [29]:
pd.read_sql_query("""
    SELECT * FROM accounts
    ORDER BY signup_date ASC, account_id ASC
    LIMIT 5;
    """, conn
)

,account_id,signup_date,country
0,145,2026-07-03,GB
1,161,2026-07-03,US
2,265,2026-07-03,GB
3,347,2026-07-03,CA
4,356,2026-07-03,DE


In [30]:
pd.read_sql_query("""
    SELECT event_type, COUNT(*) AS event_count FROM events
    GROUP BY event_type
    ORDER BY event_count DESC;
    """, conn
)

,event_type,event_count
0,click,149308
1,login,87042
2,post,63650


In [31]:
pd.read_sql_query("""
    SELECT MIN(event_time) AS first_event,
    MAX(event_time) AS last_event
    FROM events;
    """, conn
)

,first_event,last_event
0,2026-09-01 00:00:03,2026-09-30 23:59:54


In [32]:
pd.read_sql_query("""
    SELECT account_id, COUNT(*) AS event_count FROM events
    GROUP BY account_id
    ORDER BY event_count DESC, account_id ASC
    LIMIT 10;
    """, conn
)


,account_id,event_count
0,1918,25
1,5174,25
2,10301,25
3,3908,24
4,4730,24
5,8033,24
6,13259,24
7,14205,24
8,16664,24
9,16978,24


In [33]:
pd.read_sql_query("""
  SELECT
    MIN(ec) AS min_events,
    MAX(ec) AS max_events,
    AVG(ec) AS avg_events
  FROM
    (
    SELECT account_id, COUNT(*) AS ec FROM events
    GROUP BY account_id );
    """, conn
)


,min_events,max_events,avg_events
0,10,25,15.0


In [34]:
pd.read_sql_query("""
  SELECT DATE(event_time) AS event_date,
  COUNT(*) AS event_count FROM events
  GROUP BY DATE(event_time) ORDER BY DATE(event_time);
  """, conn
)

,event_date,event_count
0,2026-09-01,10017
1,2026-09-02,10327
2,2026-09-03,10169
3,2026-09-04,10420
4,2026-09-05,10478
5,2026-09-06,10169
6,2026-09-07,10175
7,2026-09-08,9838
8,2026-09-09,9748
9,2026-09-10,10096


### Click burst detection

In [35]:
pd.read_sql_query("""
  SELECT account_id, DATE(event_time) AS event_date,
  COUNT(*) AS click_count FROM events
  WHERE event_type = 'click'
  GROUP BY account_id, event_date
  ORDER BY click_count DESC, account_id, event_date
  LIMIT 10;
  """, conn
)


,account_id,event_date,click_count
0,5089,2026-09-30,15
1,8404,2026-09-13,15
2,1174,2026-09-02,14
3,1582,2026-09-01,14
4,3452,2026-09-21,14
5,3629,2026-09-01,14
6,3687,2026-09-03,14
7,4474,2026-09-16,14
8,4508,2026-09-08,14
9,4903,2026-09-08,14


In [36]:
pd.read_sql_query("""
  SELECT account_id, MIN(event_time) AS first_click,
  MAX(event_time) AS last_click,
  COUNT(*) AS click_count FROM events
  WHERE event_type = 'click'
  GROUP BY account_id
  ORDER BY click_count DESC, account_id
  LIMIT 10;
  """, conn
)


,account_id,first_click,last_click,click_count
0,2399,2026-09-02 06:01:47,2026-09-19 15:21:11,17
1,3239,2026-09-01 11:27:43,2026-09-30 12:37:31,17
2,3478,2026-09-14 08:28:10,2026-09-27 04:28:24,17
3,5089,2026-09-08 08:36:56,2026-09-30 08:12:29,17
4,7108,2026-09-07 09:47:51,2026-09-25 01:28:00,17
5,8395,2026-09-04 19:40:58,2026-09-29 19:56:28,17
6,11217,2026-09-12 08:46:41,2026-09-29 22:25:45,17
7,11327,2026-09-13 14:03:44,2026-09-29 05:16:48,17
8,12503,2026-09-04 08:11:34,2026-09-28 14:12:50,17
9,17139,2026-09-05 19:47:08,2026-09-25 14:06:31,17


In [37]:
pd.read_sql_query("""
  SELECT account_id, strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
  COUNT(*) AS click_count FROM events
  WHERE event_type = 'click'
  GROUP BY account_id, event_hour
  ORDER BY click_count DESC, account_id, event_hour
  LIMIT 10;
  """, conn
)


,account_id,event_hour,click_count
0,1174,2026-09-02 13:00:00,14
1,1582,2026-09-01 08:00:00,14
2,3452,2026-09-21 01:00:00,14
3,3629,2026-09-01 06:00:00,14
4,3687,2026-09-03 22:00:00,14
5,4474,2026-09-16 19:00:00,14
6,4508,2026-09-08 02:00:00,14
7,4903,2026-09-08 14:00:00,14
8,5089,2026-09-30 08:00:00,14
9,5815,2026-09-24 13:00:00,14


In [38]:
pd.read_sql_query("""
  SELECT account_id, strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
  COUNT(*) AS click_count FROM events
  WHERE event_type = 'click'
  GROUP BY account_id, event_hour
  HAVING click_count >= 10
  ORDER BY click_count DESC, account_id, event_hour
  ;
  """, conn
)


,account_id,event_hour,click_count
0,1174,2026-09-02 13:00:00,14
1,1582,2026-09-01 08:00:00,14
2,3452,2026-09-21 01:00:00,14
3,3629,2026-09-01 06:00:00,14
4,3687,2026-09-03 22:00:00,14
...,...,...,...
210,18878,2026-09-27 11:00:00,10
211,19239,2026-09-08 01:00:00,10
212,19260,2026-09-29 09:00:00,10
213,19633,2026-09-26 08:00:00,10


In [39]:
pd.read_sql_query("""
SELECT COUNT(DISTINCT account_id) AS flagged_accounts
FROM (
       SELECT account_id, strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       HAVING click_count >= 10)
  ;
  """, conn
)


,flagged_accounts
0,215


In [40]:
pd.read_sql_query("""
SELECT DISTINCT account_id
FROM (
       SELECT account_id, strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       HAVING click_count >= 10)
  ORDER BY account_id;
  """, conn
)

,account_id
0,66
1,313
2,432
3,452
4,480
...,...
210,19633
211,19813
212,19854
213,19981


In [41]:
pd.read_sql_query("""
SELECT DISTINCT x.account_id, signup_date, country
FROM (
       SELECT account_id, strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       HAVING click_count >= 10) x
  INNER JOIN accounts ON x.account_id = accounts.account_id
ORDER BY x.account_id;
  """, conn
)

,account_id,signup_date,country
0,66,2026-08-12,DE
1,313,2026-08-24,US
2,432,2026-07-18,GB
3,452,2026-07-07,DE
4,480,2026-08-25,GB
...,...,...,...
210,19633,2026-08-20,GB
211,19813,2026-08-30,GB
212,19854,2026-07-23,DE
213,19981,2026-08-08,IN


In [42]:
pd.read_sql_query("""
       SELECT account_id,
       strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count,
       unixepoch(MAX(event_time)) - unixepoch(MIN(event_time)) AS span_seconds
       FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       HAVING click_count >= 10 AND span_seconds <= 60
       ORDER BY span_seconds, click_count DESC, account_id, event_hour;
  """, conn
)

,account_id,event_hour,click_count,span_seconds
0,10840,2026-09-05 09:00:00,12,6
1,8453,2026-09-08 11:00:00,11,6
2,17512,2026-09-09 22:00:00,11,6
3,1854,2026-09-23 04:00:00,10,6
4,4761,2026-09-04 21:00:00,10,6
...,...,...,...,...
142,10491,2026-09-26 22:00:00,13,29
143,11217,2026-09-12 08:00:00,13,29
144,9240,2026-09-20 09:00:00,12,29
145,10953,2026-09-05 01:00:00,12,29


In [43]:
pd.read_sql_query("""
  SELECT COUNT(*) AS burst_count,
  MIN(span_seconds) AS min_span_seconds,
  MAX(span_seconds) AS max_span_seconds,
  AVG(span_seconds) AS avg_span_seconds
  FROM (
       SELECT account_id,
       strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count,
       unixepoch(MAX(event_time)) - unixepoch(MIN(event_time)) AS span_seconds
       FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       );
  """, conn
)

,burst_count,min_span_seconds,max_span_seconds,avg_span_seconds
0,143888,0,3416,7.673114


In [44]:
pd.read_sql_query("""
  SELECT
  COUNT(DISTINCT CASE WHEN click_count >=5 THEN account_id END) AS min5clicks,
  COUNT(DISTINCT CASE WHEN click_count >=10 THEN account_id END) AS min10clicks,
  COUNT(DISTINCT CASE WHEN click_count >=15 THEN account_id END) AS min15clicks
  FROM (
       SELECT account_id,
       strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count,
       unixepoch(MAX(event_time)) - unixepoch(MIN(event_time)) AS span_seconds
       FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       );
  """, conn
)

,min5clicks,min10clicks,min15clicks
0,532,215,0


In [45]:
pd.read_sql_query("""
  SELECT account_id, MAX(click_count) AS max_hourly_clicks
  FROM (
       SELECT account_id,
       strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count
       FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       HAVING click_count >= 5
       )
  GROUP BY account_id
  ORDER BY max_hourly_clicks DESC, account_id;
       """, conn
)

,account_id,max_hourly_clicks
0,1174,14
1,1582,14
2,3452,14
3,3629,14
4,3687,14
...,...,...
527,19264,5
528,19421,5
529,19648,5
530,19765,5


In [46]:
pd.read_sql_query("""
SELECT max_hourly_clicks, COUNT(account_id) AS account_count
FROM(
  SELECT account_id, MAX(click_count) AS max_hourly_clicks
  FROM (
       SELECT account_id,
       strftime('%Y-%m-%d %H:00:00', event_time) AS event_hour,
       COUNT() AS click_count
       FROM events
       WHERE event_type = 'click'
       GROUP BY account_id, event_hour
       HAVING click_count >= 5
       )
  GROUP BY account_id)
  GROUP BY max_hourly_clicks
  ORDER BY max_hourly_clicks;
       """, conn
)

,max_hourly_clicks,account_count
0,5,112
1,6,64
2,7,47
3,8,52
4,9,42
5,10,46
6,11,49
7,12,47
8,13,42
9,14,31


In [47]:
pd.read_sql_query("""
SELECT account_id, event_id, event_time,
LAG(event_time,1) OVER (PARTITION BY account_id ORDER BY event_time, event_id) AS prev_click_time
FROM events
WHERE event_type = 'click'
ORDER BY account_id, event_time, event_id
LIMIT 20;
""",   conn )

,account_id,event_id,event_time,prev_click_time
0,1,24635,2026-09-03 09:55:55,None
1,1,66711,2026-09-07 12:07:20,2026-09-03 09:55:55
2,1,66716,2026-09-07 12:07:58,2026-09-07 12:07:20
3,1,98714,2026-09-10 17:20:10,2026-09-07 12:07:58
4,1,114047,2026-09-12 06:41:26,2026-09-10 17:20:10
5,1,216976,2026-09-22 14:38:55,2026-09-12 06:41:26
6,1,228291,2026-09-23 17:53:31,2026-09-22 14:38:55
7,1,245257,2026-09-25 11:03:48,2026-09-23 17:53:31
8,1,275469,2026-09-28 12:05:56,2026-09-25 11:03:48
9,2,20799,2026-09-03 01:00:25,None


In [48]:
pd.read_sql_query("""
SELECT account_id, event_id, event_time,
LAG(event_time,1) OVER (PARTITION BY account_id ORDER BY event_time, event_id) AS prev_click_time,
unixepoch(event_time) - unixepoch(LAG(event_time,1) OVER (PARTITION BY account_id ORDER BY event_time, event_id)) AS gap_seconds
FROM events
WHERE event_type = 'click'
ORDER BY account_id, event_time, event_id
LIMIT 20;
""",   conn )

,account_id,event_id,event_time,prev_click_time,gap_seconds
0,1,24635,2026-09-03 09:55:55,None,NaN
1,1,66711,2026-09-07 12:07:20,2026-09-03 09:55:55,353485.0
2,1,66716,2026-09-07 12:07:58,2026-09-07 12:07:20,38.0
3,1,98714,2026-09-10 17:20:10,2026-09-07 12:07:58,277932.0
4,1,114047,2026-09-12 06:41:26,2026-09-10 17:20:10,134476.0
5,1,216976,2026-09-22 14:38:55,2026-09-12 06:41:26,892649.0
6,1,228291,2026-09-23 17:53:31,2026-09-22 14:38:55,98076.0
7,1,245257,2026-09-25 11:03:48,2026-09-23 17:53:31,148217.0
8,1,275469,2026-09-28 12:05:56,2026-09-25 11:03:48,262928.0
9,2,20799,2026-09-03 01:00:25,None,NaN


In [49]:
pd.read_sql_query("""
  SELECT account_id, event_id, event_time, prev_click_time, gap_seconds

  FROM (
    SELECT account_id, event_id, event_time,
    LAG(event_time,1) OVER (PARTITION BY account_id ORDER BY event_time, event_id) AS prev_click_time,
    unixepoch(event_time) - unixepoch(LAG(event_time,1) OVER (PARTITION BY account_id ORDER BY event_time, event_id)) AS gap_seconds
    FROM events
    WHERE event_type = 'click'
    )
    WHERE gap_seconds <= 10
    ORDER BY account_id, event_time, event_id
    LIMIT 20;
""",   conn )

,account_id,event_id,event_time,prev_click_time,gap_seconds
0,66,73105,2026-09-08 03:22:00,2026-09-08 03:21:59,1
1,66,73106,2026-09-08 03:22:00,2026-09-08 03:22:00,0
2,66,73107,2026-09-08 03:22:00,2026-09-08 03:22:00,0
3,66,73108,2026-09-08 03:22:00,2026-09-08 03:22:00,0
4,66,73109,2026-09-08 03:22:01,2026-09-08 03:22:00,1
5,66,73110,2026-09-08 03:22:01,2026-09-08 03:22:01,0
6,66,73111,2026-09-08 03:22:03,2026-09-08 03:22:01,2
7,66,73112,2026-09-08 03:22:04,2026-09-08 03:22:03,1
8,66,73113,2026-09-08 03:22:06,2026-09-08 03:22:04,2
9,66,73114,2026-09-08 03:22:08,2026-09-08 03:22:06,2


In [50]:
pd.read_sql_query("""
  SELECT account_id, event_id, event_time, tenth_click,
  unixepoch(tenth_click) - unixepoch(event_time) AS tenth_click_gap
  FROM (
    SELECT account_id, event_id, event_time,
    NTH_VALUE(event_time,10) OVER (PARTITION BY account_id ORDER BY event_time, event_id
    ROWS BETWEEN 0 PRECEDING AND UNBOUNDED FOLLOWING) AS tenth_click
    FROM events
    WHERE event_type = 'click'
    )
    WHERE tenth_click_gap <= 60
    ORDER BY tenth_click_gap, account_id, event_time, event_id
    ;
""",   conn )

,account_id,event_id,event_time,tenth_click,tenth_click_gap
0,15279,21389,2026-09-03 02:21:42,2026-09-03 02:21:44,2
1,15279,21390,2026-09-03 02:21:42,2026-09-03 02:21:45,3
2,15279,21391,2026-09-03 02:21:43,2026-09-03 02:21:46,3
3,8713,298973,2026-09-30 21:24:16,2026-09-30 21:24:20,4
4,9674,242468,2026-09-25 04:11:12,2026-09-25 04:11:16,4
...,...,...,...,...,...
451,15062,135358,2026-09-14 10:57:48,2026-09-14 10:58:45,57
452,2085,273823,2026-09-28 08:07:20,2026-09-28 08:08:18,58
453,16042,200668,2026-09-20 23:52:47,2026-09-20 23:53:46,59
454,9344,27028,2026-09-03 15:42:34,2026-09-03 15:43:34,60


In [51]:
ten_click_flags_60 = pd.read_sql_query("""
  SELECT account_id AS flagged_accounts,
  MIN(unixepoch(tenth_click) - unixepoch(event_time)) AS strongest_burst_span
  FROM (
    SELECT account_id, event_id, event_time,
    NTH_VALUE(event_time,10) OVER (PARTITION BY account_id ORDER BY event_time, event_id
    ROWS BETWEEN 0 PRECEDING AND UNBOUNDED FOLLOWING) AS tenth_click
    FROM events
    WHERE event_type = 'click'
    )
    GROUP BY account_id
    HAVING strongest_burst_span <= 60
    ORDER BY strongest_burst_span, account_id
    ;
""",   conn )
ten_click_flags_60

,flagged_accounts,strongest_burst_span
0,15279,2
1,8713,4
2,9674,4
3,12433,4
4,13739,4
...,...,...
156,11308,55
157,3721,56
158,2085,58
159,9344,60


In [52]:
ten_click_flags_120 = pd.read_sql_query("""
  SELECT account_id AS flagged_accounts,
  MIN(unixepoch(tenth_click) - unixepoch(event_time)) AS strongest_burst_span
  FROM (
    SELECT account_id, event_id, event_time,
    NTH_VALUE(event_time,10) OVER (PARTITION BY account_id ORDER BY event_time, event_id
    ROWS BETWEEN 0 PRECEDING AND UNBOUNDED FOLLOWING) AS tenth_click
    FROM events
    WHERE event_type = 'click'
    )
    GROUP BY account_id
    HAVING strongest_burst_span <= 120
    ORDER BY strongest_burst_span, account_id
    ;
""",   conn )
ten_click_flags_120

,flagged_accounts,strongest_burst_span
0,15279,2
1,8713,4
2,9674,4
3,12433,4
4,13739,4
...,...,...
213,4814,107
214,18878,107
215,6949,110
216,3474,111


In [53]:
five_click_flags_60 = pd.read_sql_query("""
  SELECT account_id AS flagged_accounts,
  MIN(unixepoch(tenth_click) - unixepoch(event_time)) AS strongest_burst_span
  FROM (
    SELECT account_id, event_id, event_time,
    NTH_VALUE(event_time,5) OVER (PARTITION BY account_id ORDER BY event_time, event_id
    ROWS BETWEEN 0 PRECEDING AND UNBOUNDED FOLLOWING) AS tenth_click
    FROM events
    WHERE event_type = 'click'
    )
    GROUP BY account_id
    HAVING strongest_burst_span <= 60
    ORDER BY strongest_burst_span, account_id
    ;
""",   conn )
five_click_flags_60

,flagged_accounts,strongest_burst_span
0,5162,0
1,15279,0
2,16271,0
3,16677,0
4,66,1
...,...,...
426,10096,52
427,9914,53
428,11403,53
429,4370,55


In [54]:
five_click_flags_30 = pd.read_sql_query("""
  SELECT account_id AS flagged_accounts,
  MIN(unixepoch(tenth_click) - unixepoch(event_time)) AS strongest_burst_span
  FROM (
    SELECT account_id, event_id, event_time,
    NTH_VALUE(event_time,5) OVER (PARTITION BY account_id ORDER BY event_time, event_id
    ROWS BETWEEN 0 PRECEDING AND UNBOUNDED FOLLOWING) AS tenth_click
    FROM events
    WHERE event_type = 'click'
    )
    GROUP BY account_id
    HAVING strongest_burst_span <= 30
    ORDER BY strongest_burst_span, account_id
    ;
""",   conn )
five_click_flags_30

,flagged_accounts,strongest_burst_span
0,5162,0
1,15279,0
2,16271,0
3,16677,0
4,66,1
...,...,...
370,4212,29
371,4801,29
372,6701,29
373,2971,30


In [55]:
eight_click_flags_60 = pd.read_sql_query("""
  SELECT account_id AS flagged_accounts,
  MIN(unixepoch(tenth_click) - unixepoch(event_time)) AS strongest_burst_span
  FROM (
    SELECT account_id, event_id, event_time,
    NTH_VALUE(event_time,8) OVER (PARTITION BY account_id ORDER BY event_time, event_id
    ROWS BETWEEN 0 PRECEDING AND UNBOUNDED FOLLOWING) AS tenth_click
    FROM events
    WHERE event_type = 'click'
    )
    GROUP BY account_id
    HAVING strongest_burst_span <= 60
    ORDER BY strongest_burst_span, account_id
    ;
""",   conn )
eight_click_flags_60

,flagged_accounts,strongest_burst_span
0,4508,1
1,15279,1
2,8713,2
3,16677,2
4,17906,2
...,...,...
243,17472,57
244,11439,58
245,4968,59
246,16878,59


### Click burst evaluation

In [56]:
ground_truth_sql = pd.read_csv(DATA_DIR / "ground_truth.csv")
print(ground_truth_sql.columns.tolist())

['account_id', 'pattern']


In [57]:
ground_truth_sql.to_sql(
    "ground_truth",
    conn,
    index=False,
    if_exists="replace"
)

20000

In [58]:
pd.read_sql_query("""
  SELECT pattern, COUNT(*) AS account_count
  FROM ground_truth
  GROUP BY pattern;
""",   conn )


,pattern,account_count
0,automated_clicking,300
1,legitimate,19000
2,spam,400
3,takeover,300


In [59]:
eight_click_flags_60.to_sql(
    "eight_click_flags_60",
    conn,
    index=False,
    if_exists="replace"
)

five_click_flags_60.to_sql(
    "five_click_flags_60",
    conn,
    index=False,
    if_exists="replace"
)

five_click_flags_30.to_sql(
    "five_click_flags_30",
    conn,
    index=False,
    if_exists="replace"
)

ten_click_flags_60.to_sql(
    "ten_click_flags_60",
    conn,
    index=False,
    if_exists="replace"
)

ten_click_flags_120.to_sql(
    "ten_click_flags_120",
    conn,
    index=False,
    if_exists="replace"
)

218

In [60]:
pd.read_sql_query("""
  SELECT t1.flagged_accounts,
  t2.pattern
  FROM eight_click_flags_60 t1
  LEFT JOIN ground_truth t2
  ON t2.account_id = t1.flagged_accounts;
  """, conn
)

,flagged_accounts,pattern
0,4508,legitimate
1,15279,automated_clicking
2,8713,automated_clicking
3,16677,legitimate
4,17906,automated_clicking
...,...,...
243,17472,legitimate
244,11439,legitimate
245,4968,legitimate
246,16878,legitimate


In [61]:
result_8_60 = pd.read_sql_query("""
  SELECT pattern, COUNT(*) AS flagged_count
  FROM (
      SELECT t1.flagged_accounts,
      t2.pattern
      FROM eight_click_flags_60 t1
      LEFT JOIN ground_truth t2
      ON t2.account_id = t1.flagged_accounts)
      GROUP BY pattern;
  """, conn
)
result_8_60

,pattern,flagged_count
0,automated_clicking,142
1,legitimate,106


In [62]:
result_5_60 = pd.read_sql_query("""
  SELECT pattern, COUNT(*) AS flagged_count
  FROM (
      SELECT t1.flagged_accounts,
      t2.pattern
      FROM five_click_flags_60 t1
      LEFT JOIN ground_truth t2
      ON t2.account_id = t1.flagged_accounts)
      GROUP BY pattern;
  """, conn
)
result_5_60

,pattern,flagged_count
0,automated_clicking,257
1,legitimate,174


In [63]:
result_5_30 = pd.read_sql_query("""
  SELECT pattern, COUNT(*) AS flagged_count
  FROM (
      SELECT t1.flagged_accounts,
      t2.pattern
      FROM five_click_flags_30 t1
      LEFT JOIN ground_truth t2
      ON t2.account_id = t1.flagged_accounts)
      GROUP BY pattern;
  """, conn
)
result_5_30

,pattern,flagged_count
0,automated_clicking,219
1,legitimate,156


In [64]:
result_10_60 = pd.read_sql_query("""
  SELECT pattern, COUNT(*) AS flagged_count
  FROM (
      SELECT t1.flagged_accounts,
      t2.pattern
      FROM ten_click_flags_60 t1
      LEFT JOIN ground_truth t2
      ON t2.account_id = t1.flagged_accounts)
      GROUP BY pattern;
  """, conn
)
result_10_60

,pattern,flagged_count
0,automated_clicking,99
1,legitimate,62


In [65]:
result_10_120 = pd.read_sql_query("""
  SELECT pattern, COUNT(*) AS flagged_count
  FROM (
      SELECT t1.flagged_accounts,
      t2.pattern
      FROM ten_click_flags_120 t1
      LEFT JOIN ground_truth t2
      ON t2.account_id = t1.flagged_accounts)
      GROUP BY pattern;
  """, conn
)
result_10_120

,pattern,flagged_count
0,automated_clicking,131
1,legitimate,87


In [66]:
candidate_results = {
    "5 clicks / 30 sec": result_5_30,
    "5 clicks / 60 sec": result_5_60,
    "8 clicks / 60 sec": result_8_60,
    "10 clicks / 60 sec": result_10_60,
    "10 clicks / 120 sec": result_10_120
}

rows = []

for rule, result in candidate_results.items():
    counts = result.set_index("pattern")["flagged_count"]

    flagged = counts.sum()
    caught = counts.get("automated_clicking", 0)
    legitimate = counts.get("legitimate", 0)

    rows.append({
        "click_flag_rule": rule,
        "flagged": flagged,
        "caught": caught,
        "legit_false_alarms": legitimate,
        "other_pattern_flags": flagged - caught - legitimate,
        "precision%": 100 * caught / flagged if flagged else np.nan,
        "recall%": 100 * caught / 300,
        "review%": 100 * flagged / 20000
    })

click_burst_eval = pd.DataFrame(rows)
click_burst_eval

,click_flag_rule,flagged,caught,legit_false_alarms,other_pattern_flags,precision%,recall%,review%
0,5 clicks / 30 sec,375,219,156,0,58.400000,73.000000,1.875
1,5 clicks / 60 sec,431,257,174,0,59.628770,85.666667,2.155
2,8 clicks / 60 sec,248,142,106,0,57.258065,47.333333,1.240
3,10 clicks / 60 sec,161,99,62,0,61.490683,33.000000,0.805
4,10 clicks / 120 sec,218,131,87,0,60.091743,43.666667,1.090


In [67]:
pd.DataFrame({
    "account_id": legit_click_ids
}).to_sql(
    "legit_click_accounts",
    conn,
    index=False,
    if_exists="replace"
)


200

In [68]:
five_click_flags_60

,flagged_accounts,strongest_burst_span
0,5162,0
1,15279,0
2,16271,0
3,16677,0
4,66,1
...,...,...
426,10096,52
427,9914,53
428,11403,53
429,4370,55


In [69]:
pd.read_sql_query("""
  SELECT COUNT(DISTINCT t1.flagged_accounts) AS flagged_genuine_click_accounts
  FROM five_click_flags_60 t1
  INNER JOIN legit_click_accounts t2
  ON t2.account_id = t1.flagged_accounts;
  """, conn
)



,flagged_genuine_click_accounts
0,174


In [70]:
pd.read_sql_query("""
SELECT account_id AS unflagged_accounts FROM ground_truth
WHERE pattern = 'automated_clicking' AND account_id NOT IN
(
    SELECT flagged_accounts
    FROM five_click_flags_60
) ORDER BY unflagged_accounts
  ;
  """, conn
)

,unflagged_accounts
0,204
1,327
2,818
3,1439
4,2965
5,3027
6,4926
7,5903
8,5950
9,5987


In [71]:
pd.read_sql_query("""
  SELECT t1.unflagged_accounts,
  COUNT(t2.account_id) AS total_clicks
  FROM (
    SELECT account_id AS unflagged_accounts FROM ground_truth
WHERE pattern = 'automated_clicking' AND account_id NOT IN
(SELECT flagged_accounts FROM five_click_flags_60)
) t1
LEFT JOIN events t2
ON t2.account_id = t1.unflagged_accounts AND t2.event_type = 'click'
GROUP BY t1.unflagged_accounts
ORDER BY account_id;
""", conn
  )

,unflagged_accounts,total_clicks
0,204,5
1,327,9
2,818,12
3,1439,9
4,2965,12
5,3027,6
6,4926,10
7,5903,9
8,5950,10
9,5987,6


In [72]:
pd.read_sql_query("""
SELECT unflagged_accounts,
  COUNT(event_id) AS total_clicks,
  MIN(unixepoch(fifth_click) - unixepoch(event_time)) AS min_5_click_span
  FROM (
  SELECT t1.unflagged_accounts, t2.event_id, t2.event_time,
  NTH_VALUE(event_time,5) OVER(PARTITION BY unflagged_accounts
  ORDER BY event_time, event_id
  ROWS BETWEEN 0 PRECEDING AND UNBOUNDED FOLLOWING) AS fifth_click
  FROM (
    SELECT account_id AS unflagged_accounts FROM ground_truth
    WHERE pattern = 'automated_clicking' AND account_id NOT IN
    (SELECT flagged_accounts FROM five_click_flags_60)
    ) t1
    LEFT JOIN events t2
    ON t2.account_id = t1.unflagged_accounts AND t2.event_type = 'click')

GROUP BY unflagged_accounts
;
""", conn

)

,unflagged_accounts,total_clicks,min_5_click_span
0,204,5,160097
1,327,9,109
2,818,12,230925
3,1439,9,24402
4,2965,12,78
5,3027,6,43598
6,4926,10,103
7,5903,9,973843
8,5950,10,72
9,5987,6,94


#### Click-burst detection: findings and limitations

Five rolling click-burst rules were compared using precision, recall and review workload.

The preferred candidate for prioritising recall was **at least 5 clicks within 60 seconds**. It flagged 431 accounts (2.155% of all accounts), catching 257 of 300 automated-clicking accounts:

- Precision: **59.63%**
- Recall: **85.67%**
- Legitimate false alarms: **174**
- Missed automated-clicking accounts: **43**

The 10-click/60-second rule achieved the highest precision (61.49%), but recalled only 33% of target accounts. The preferred candidate accepts more review workload and legitimate false alarms to achieve greater coverage.

#### False-alarm review

All 174 legitimate false alarms belonged to the 200 genuine click-spike lookalikes: the rule flagged 87% of that group. No other legitimate accounts, spam accounts or takeover accounts were flagged.

This shows that rapid clicking alone does not reliably distinguish automated activity from genuine click spikes in this simulation.

#### Missed-account review

All 43 missed automated-clicking accounts had at least five clicks overall, but none had five consecutive clicks within 60 seconds. Their minimum five-click spans ranged from 66 to 1,836,254 seconds.

The rule therefore missed target accounts whose activity did not meet its required combination of click volume and timing. This does not rule out shorter bursts containing fewer than five clicks.

#### Interpretation and limitations

Flags identify accounts for manual review, not confirmed abuse or automatic enforcement.

Rules were developed, compared and selected on the same synthetic dataset, with knowledge of the generator's patterns. These are in-sample demonstration results, not evidence of effectiveness on unseen or real-world data. The preferred threshold reflects an illustrative preference for recall; real operating decisions would also require review capacity and investigation costs.

### Spam post detection

In [73]:
pd.read_sql_query("""
  SELECT account_id, event_time, ip_address,
  RTRIM(RTRIM(ip_address, '0123456789'), '.') AS ip_block
  FROM events
  WHERE event_type = 'post'
  ORDER BY event_time, event_id
  LIMIT 20;
""",   conn )

,account_id,event_time,ip_address,ip_block
0,18436,2026-09-01 00:00:58,10.231.160.103,10.231.160
1,3845,2026-09-01 00:01:01,10.185.241.4,10.185.241
2,18685,2026-09-01 00:01:18,10.139.12.208,10.139.12
3,435,2026-09-01 00:01:22,10.223.52.90,10.223.52
4,15805,2026-09-01 00:01:50,10.137.9.208,10.137.9
5,1712,2026-09-01 00:02:14,10.148.35.233,10.148.35
6,19187,2026-09-01 00:02:28,10.218.84.201,10.218.84
7,10386,2026-09-01 00:03:24,10.95.191.45,10.95.191
8,9218,2026-09-01 00:03:27,10.225.239.220,10.225.239
9,19345,2026-09-01 00:03:53,10.252.212.115,10.252.212


In [87]:
conn.execute("""
CREATE TEMP TABLE IF NOT EXISTS posts AS
SELECT
    event_id,
    account_id,
    event_time,
    RTRIM(RTRIM(ip_address, '0123456789'), '.') AS ip_block
FROM events
WHERE event_type = 'post';
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_posts_block_time
ON posts (ip_block, event_time);
""")

spam_windows = pd.read_sql_query("""
SELECT
    t1.event_id AS anchor_event_id,
    t1.ip_block,
    t1.event_time AS anchor_time,
    COUNT(DISTINCT t2.account_id) AS distinct_accounts
FROM posts t1, posts t2
    WHERE t2.ip_block = t1.ip_block
   AND t2.event_time BETWEEN t1.event_time AND DATETIME(t1.event_time, '+100 seconds')
GROUP BY
    t1.event_id,
    t1.ip_block
HAVING COUNT(DISTINCT t2.account_id) >= 15
ORDER BY anchor_time, anchor_event_id;
""", conn)

spam_windows.to_sql(
    "spam_windows",
    conn,
    index=False,
    if_exists="replace"
)

spam_windows

,anchor_event_id,ip_block,anchor_time,distinct_accounts
0,15548,10.250.20,2026-09-02 13:01:52,15
1,15549,10.250.20,2026-09-02 13:01:55,16
2,15551,10.250.20,2026-09-02 13:02:04,17
3,15553,10.250.20,2026-09-02 13:02:07,17
4,15554,10.250.20,2026-09-02 13:02:07,17
...,...,...,...,...
972,267116,10.250.1,2026-09-27 15:40:30,15
973,267117,10.250.1,2026-09-27 15:40:32,15
974,267118,10.250.1,2026-09-27 15:40:32,15
975,267119,10.250.1,2026-09-27 15:40:40,15


In [89]:
spam_post_flags_15p_100s = pd.read_sql_query("""
    SELECT DISTINCT p.account_id AS flagged_accounts
    FROM spam_windows w
    LEFT JOIN posts p
    ON p.ip_block = w.ip_block
    AND p.event_time BETWEEN w.anchor_time AND DATETIME(w.anchor_time, '+100 seconds')
    ORDER BY flagged_accounts;
     """, conn    )


In [90]:
spam_post_flags_15p_100s.to_sql(
    "spam_post_flags_15p_100s",
    conn,
    index=False,
    if_exists="replace"
)

402

### Spam post evaluation

In [91]:
spam_result_15_100 = pd.read_sql_query("""
  SELECT g.pattern,
         COUNT(s.flagged_accounts) AS flagged_count
  FROM spam_post_flags_15p_100s s
  LEFT JOIN ground_truth g
  ON s.flagged_accounts = g.account_id
  GROUP BY g.pattern;
""",   conn )

In [92]:
spam_result_15_100

,pattern,flagged_count
0,legitimate,30
1,spam,372


In [93]:
spam_candidates = {
    "10 acc / 120 sec": spam_result_10_120,
    "15 acc / 120 sec": spam_result_15_120,
    "16 acc / 120 sec": spam_result_16_120,
    "15 acc / 100 sec": spam_result_15_100
}

rows = []

for rule, result in spam_candidates.items():
    counts = result.set_index("pattern")["flagged_count"]

    flagged = counts.sum()
    caught = counts.get("spam", 0)
    legitimate = counts.get("legitimate", 0)

    rows.append({
        "spam_flag_rule": rule,
        "flagged": flagged,
        "caught": caught,
        "legit_false_alarms": legitimate,
        "other_pattern_flags": flagged - caught - legitimate,
        "precision%": 100 * caught / flagged if flagged else np.nan,
        "recall%": 100 * caught / 400,
        "review%": 100 * flagged / 20000
    })

spam_post_eval = pd.DataFrame(rows)

spam_post_eval

,spam_flag_rule,flagged,caught,legit_false_alarms,other_pattern_flags,precision%,recall%,review%
0,10 acc / 120 sec,783,400,383,0,51.085568,100.00,3.915
1,15 acc / 120 sec,483,400,83,0,82.815735,100.00,2.415
2,16 acc / 120 sec,408,375,33,0,91.911765,93.75,2.040
3,15 acc / 100 sec,402,372,30,0,92.537313,93.00,2.010


In [94]:
pd.DataFrame({
    "account_id": legit_shared_ids
}).to_sql(
    "legit_shared_accounts",
    conn,
    index=False,
    if_exists="replace"
)


400

In [95]:
spam_post_flags_15p_120s

,flagged_accounts
0,49
1,77
2,83
3,91
4,127
...,...
478,19879
479,19937
480,19945
481,19946


In [96]:
pd.read_sql_query("""
  SELECT COUNT(DISTINCT t1.flagged_accounts) AS flagged_genuine_shared_accounts
  FROM spam_post_flags_15p_120s t1
  INNER JOIN legit_shared_accounts t2
  ON t2.account_id = t1.flagged_accounts;
  """, conn
)



,flagged_genuine_shared_accounts
0,83


#### Shared-network posting: findings and limitations

The rule identifies accounts posting from the same /24 IP block within a rolling time window. Each account is flagged once, regardless of how many qualifying windows it participates in.

Four candidates are shown in the comparison table; additional settings were explored during development.

The preferred candidate for prioritising recall requires at least
15 distinct accounts within 120 seconds. It flagged 483 accounts:
all 400 spam accounts and 83 legitimate accounts. Precision was 82.82%, recall was 100%, and review share was 2.415%. No takeover or
automated-clicking accounts were flagged.

All 83 legitimate false alarms belonged to the 400 shared-network
lookalikes (20.75% of that group). No other legitimate accounts were flagged. This illustrates the rule's difficulty distinguishing
coordinated spam from legitimate shared-network posting.

Requiring 16 accounts within 120 seconds increased precision to 91.91% but reduced recall to 93.75%. Requiring 15 accounts within 100 seconds achieved 92.54% precision and 93% recall. These alternatives reduce review workload and false alarms at the cost of missed spam accounts.

Shared-network posting bursts are a review signal, not proof of spam.
Legitimate shared-network activity can satisfy the same conditions.
The /24 grouping is an illustrative assumption.

Thresholds were developed and selected on the same synthetic dataset.
These in-sample results do not establish effectiveness on unseen or
real-world activity.

### Account Takeover Detection

In [97]:
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_events_account_time_country
ON events (account_id, event_time, country);
""")

unfamiliar_logins = pd.read_sql_query("""
SELECT
    l.account_id,
    l.event_id AS login_event_id,
    l.event_time AS login_time,
    l.country
FROM events AS l
WHERE l.event_type = 'login'
  AND (
      SELECT COUNT(*)
      FROM events AS h
      WHERE h.account_id = l.account_id
        AND h.event_time < l.event_time
  ) >= 5
  AND NOT EXISTS (
      SELECT 1
      FROM events AS h
      WHERE h.account_id = l.account_id
        AND h.event_time < l.event_time
        AND h.country = l.country
  )
ORDER BY l.account_id, l.event_time, l.event_id;
""", conn)

unfamiliar_logins.to_sql(
    "unfamiliar_logins",
    conn,
    index=False,
    if_exists="replace"
)

unfamiliar_logins

,account_id,login_event_id,login_time,country
0,31,182582,2026-09-19 04:33:52,GB
1,34,217407,2026-09-22 15:38:57,US
2,41,242191,2026-09-25 03:31:22,CA
3,43,152998,2026-09-16 05:34:13,IN
4,103,90531,2026-09-09 21:54:01,GB
...,...,...,...,...
495,19714,147755,2026-09-15 16:44:53,US
496,19742,90331,2026-09-09 21:26:09,DE
497,19765,94481,2026-09-10 07:36:17,DE
498,19778,169719,2026-09-17 22:12:32,DE


In [98]:
takeover_activity = pd.read_sql_query("""
  SELECT ul.account_id, ul.login_event_id, ul.login_time, ul.country,
         COUNT(CASE
               WHEN e.event_time < ul.login_time AND e.event_time >= datetime(ul.login_time, '-1 hour')
               THEN e.event_id END) AS prior_hour_events,
         COUNT(CASE
               WHEN e.event_time > ul.login_time AND e.event_time <= datetime(ul.login_time, '+1 hour')
               THEN e.event_id END) AS following_hour_events
  FROM unfamiliar_logins ul
  LEFT JOIN events e
  ON e.account_id = ul.account_id
  AND e.event_time BETWEEN DATETIME(ul.login_time, '-1 hour') AND DATETIME(ul.login_time, '+1 hour')
  GROUP BY ul.login_event_id
  ORDER BY ul.account_id, ul.login_event_id;
  """, conn
)

takeover_activity.to_sql(
    "takeover_activity", conn,
    index=False, if_exists="replace"
)

takeover_activity

,account_id,login_event_id,login_time,country,prior_hour_events,following_hour_events
0,31,182582,2026-09-19 04:33:52,GB,0,7
1,34,217407,2026-09-22 15:38:57,US,0,6
2,41,242191,2026-09-25 03:31:22,CA,0,8
3,43,152998,2026-09-16 05:34:13,IN,0,0
4,103,90531,2026-09-09 21:54:01,GB,0,7
...,...,...,...,...,...,...
495,19714,147755,2026-09-15 16:44:53,US,0,9
496,19742,90331,2026-09-09 21:26:09,DE,0,11
497,19765,94481,2026-09-10 07:36:17,DE,0,12
498,19778,169719,2026-09-17 22:12:32,DE,0,5


In [104]:
takeover_flags_8 = pd.read_sql_query("""
    SELECT account_id AS flagged_accounts
    FROM takeover_activity
    WHERE following_hour_events - prior_hour_events >= 8
    ORDER BY flagged_accounts;
""", conn
)

takeover_flags_8.to_sql(
    "takeover_flags_8", conn,
    index=False, if_exists="replace"
)

182

### Account Takeover Evaluation

In [105]:
takeover_result_8 = pd.read_sql_query("""

  SELECT g.pattern, COUNT(tf.flagged_accounts) AS flagged_count
  FROM takeover_flags_8 tf
  LEFT JOIN ground_truth g
  ON g.account_id = tf.flagged_accounts
  GROUP BY g.pattern;
""", conn
)

takeover_result_8


,pattern,flagged_count
0,legitimate,27
1,takeover,155


In [106]:
takeover_candidates = {
    "Activity gain >= 4": takeover_result_4,
    "Activity gain >= 5": takeover_result_5,
    "Activity gain >= 8": takeover_result_8
}

rows = []

for rule, result in takeover_candidates.items():
    counts = result.set_index("pattern")["flagged_count"]

    flagged = counts.sum()
    caught = counts.get("takeover", 0)
    legitimate = counts.get("legitimate", 0)

    rows.append({
        "takeover_flag_rule": rule,
        "flagged": flagged,
        "caught": caught,
        "legit_false_alarms": legitimate,
        "other_pattern_flags": flagged - caught - legitimate,
        "precision%": 100 * caught / flagged if flagged else np.nan,
        "recall%": 100 * caught / 300,
        "review%": 100 * flagged / 20000
    })

takeover_eval = pd.DataFrame(rows)
takeover_eval

,takeover_flag_rule,flagged,caught,legit_false_alarms,other_pattern_flags,precision%,recall%,review%
0,Activity gain >= 4,348,298,50,0,85.632184,99.333333,1.740
1,Activity gain >= 5,337,288,49,0,85.459941,96.000000,1.685
2,Activity gain >= 8,182,155,27,0,85.164835,51.666667,0.910


In [107]:
pd.DataFrame({
    "account_id": legit_travel_ids
}).to_sql(
    "legit_travel_accounts",
    conn,
    index=False,
    if_exists="replace"
)


200

In [108]:
pd.read_sql_query("""
    SELECT COUNT(t1.flagged_accounts) AS flagged_genuine_travel_accounts
    FROM takeover_flags_4 t1
    INNER JOIN legit_travel_accounts t2
    ON t2.account_id = t1.flagged_accounts;
""", conn
)


,flagged_genuine_travel_accounts
0,50


In [109]:
pd.read_sql_query("""
    SELECT account_id AS unflagged_accounts
    FROM ground_truth
    WHERE pattern = 'takeover' AND account_id NOT IN
    (SELECT flagged_accounts FROM takeover_flags_4)
    ORDER BY unflagged_accounts;
""", conn
)

,unflagged_accounts
0,5336
1,12705


In [110]:
pd.read_sql_query("""
    SELECT t1.* FROM takeover_activity t1
    JOIN (SELECT account_id AS unflagged_accounts
    FROM ground_truth
    WHERE pattern = 'takeover' AND account_id NOT IN
    (SELECT flagged_accounts FROM takeover_flags_4)) t2
    ON t1.account_id = t2.unflagged_accounts;
""", conn
)

,account_id,login_event_id,login_time,country,prior_hour_events,following_hour_events
0,5336,89992,2026-09-09 20:39:28,CA,0,3
1,12705,112610,2026-09-12 03:06:26,GB,0,3


#### Account-takeover detection: findings and limitations

Candidate logins required at least five strictly earlier events and a country absent from the account's earlier event history.

For each candidate, activity was counted in the preceding and following hour. Events at the login timestamp were excluded from both counts.
Activity gain was defined as following-hour events minus preceding-hour events.

Among the tested cutoffs of 4, 5 and 8, a gain of at least 4 was preferred for prioritising recall. It flagged 348 accounts, catching 298 of 300 takeover accounts, with 50 legitimate false alarms:

- Precision: 85.63%
- Recall: 99.33%
- Review share: 1.74%

No spam or automated-clicking accounts were flagged.

All 50 legitimate false alarms belonged to the 200 travel lookalikes (25% of that group). The rule therefore confuses some legitimate travel and increased activity with takeover.

The two missed takeover accounts passed the candidate-login checks but each had an activity gain of only 3, below the selected cutoff.

The preceding hour is a sparse baseline in this simulation, so the rule largely identifies activity following an unfamiliar-country login.
It does not establish unusual activity relative to a robust long-term behavioural baseline.

The full following-hour count is available only after that hour ends.
This is delayed detection, not an alert available at login time.

Thresholds were developed and selected on the same synthetic dataset.
These in-sample results do not establish effectiveness on unseen or
real-world data. Flags support manual review, not automatic enforcement.

## Comparison Table

In [115]:
eval_tables = {
    "Automated clicking": click_burst_eval,
    "Spam": spam_post_eval,
    "Takeover":  takeover_eval
}
rule_columns = {
    "Automated clicking": "click_flag_rule",
    "Spam": "spam_flag_rule",
    "Takeover": "takeover_flag_rule"
}
rules = {
    "Automated clicking": '5 clicks / 60 sec',
    "Spam": '15 acc / 120 sec',
    "Takeover": 'Activity gain >= 4'
}
selected_rows = []
for pattern, eval_df in eval_tables.items():
    eval_row = eval_df.loc[eval_df[rule_columns[pattern]] == rules[pattern]].copy()
    eval_row.insert(0, "pattern", pattern)
    eval_row = eval_row.rename(columns={rule_columns[pattern]: "selected_rule"})
    selected_rows.append(eval_row)

selected_rules_eval = pd.concat(selected_rows, ignore_index=True)
display(selected_rules_eval.round(2))

,pattern,selected_rule,flagged,caught,legit_false_alarms,other_pattern_flags,precision%,recall%,review%
0,Automated clicking,5 clicks / 60 sec,431,257,174,0,59.63,85.67,2.15
1,Spam,15 acc / 120 sec,483,400,83,0,82.82,100.00,2.42
2,Takeover,Activity gain >= 4,348,298,50,0,85.63,99.33,1.74


In [116]:
selected_rules_eval.to_csv(
    DATA_DIR / "selected_rules_eval.csv",
    index=False
)